# 🐧 WSL Ubuntu 24.04 — Fixing `python3-venv` (no install candidate) + Creating a Virtual Environment (venv)

This notebook is a **step-by-step guide** based on your real log (Ubuntu **24.04.3 LTS — Noble**, Python **3.12.3**).

You'll learn:
- Why `python3-venv` / `python3.12-venv` can be “not available”
- How enabling the **Universe** repository fixes it
- How to create and use a **virtual environment** (`.venv`)
- Why `PIP FREEZE` failed but `pip freeze` worked (case sensitivity)
- Why `pip freeze` can show **nothing** (and why that can be OK)

> ✅ Copy/paste the terminal commands from the code cells (they’re meant to be run in a **terminal**, not inside Python).


## 0) Your starting point (the situation)

You tried:

- `sudo apt install -y python3-venv`  → **no installation candidate**
- `sudo apt install -y python3.12-venv` → **no installation candidate**

But:
- `python3 --version` returned **Python 3.12.3**
- `cat /etc/os-release` showed **Ubuntu 24.04.3 LTS (noble)**

So the key question is:  
**Why is Python 3.12 installed, but the venv packages are “not available”?**


## 1) Why this happens on WSL Ubuntu (and on minimal installs)

### What “no installation candidate” usually means
APT is your package manager. When it says:
> `Package ... has no installation candidate`

It usually means:

1. **Your repositories are incomplete** (most common)
2. The package exists but is **not enabled** (e.g., `universe` is disabled)
3. You're using a minimal image where only `main` is enabled

### Why it matters for `python3-venv`
On Ubuntu, the `venv` support is packaged and often lives in repos that are not enabled by default in minimal installs.

✅ Fix: enable `universe` + update the package index, then install again.


## 2) Verify OS + Python version (sanity checks)

Run these to confirm what system you have.


In [ ]:
# Run in terminal:
cat /etc/os-release
python3 --version
which python3


## 3) Fix APT sources by enabling the `universe` repository

### Why `universe`?
Ubuntu splits packages into “components”:
- `main` (officially supported)
- `universe` (community-maintained, widely used)
- `restricted`, `multiverse` (special licensing / drivers)

A lot of developer tooling (including some Python packaging utilities) can depend on `universe`.

✅ Your successful path was:
1. Install repo management helpers
2. Enable `universe`
3. Refresh indexes (`apt update`)


In [ ]:
# Run in terminal:
sudo apt update

# Helps manage repositories (provides add-apt-repository)
sudo apt install -y software-properties-common

# Enable "universe" packages
sudo add-apt-repository -y universe

# Refresh package lists after adding new repo components
sudo apt update


## 4) Install venv support (now it works)

After enabling `universe`, `python3-venv` becomes available again.

### What does `python3-venv` install?
- It ensures the `venv` module is present for your Python
- Under the hood, on Ubuntu 24.04, it can pull in `python3.12-venv` too

✅ Install:


In [ ]:
# Run in terminal:
sudo apt install -y python3-venv


## 5) Create your virtual environment (venv)

### What is a venv?
A **virtual environment** is an isolated folder that contains:
- its own Python interpreter references
- its own `pip`
- its own installed packages

This prevents:
- breaking system Python
- dependency conflicts across projects
- “works on my machine” issues

### Standard convention
Use `.venv/` at the project root:
- easy to ignore in git
- VS Code detects it automatically


In [ ]:
# Run in terminal (from your project folder):
python3 -m venv .venv


## 6) Activate the venv (Linux / WSL)

Activation changes your shell so that:
- `python` points to `.venv/bin/python`
- `pip` points to `.venv/bin/pip`

✅ Activate:


In [ ]:
# Run in terminal:
source .venv/bin/activate


### Quick check: are you inside the venv?
You should see `(.venv)` at the beginning of your prompt, like:

`(.venv) camil@patate:~/business_data_management$`

Also validate with:


In [ ]:
# Run in terminal:
which python
python --version
which pip
pip --version


## 7) Upgrade pip inside the venv

You ran:
- `python -m pip install -U pip`

This is best practice, because it guarantees pip belongs to the current Python.

✅ Upgrade pip:


In [ ]:
# Run in terminal (while venv is activated):
python -m pip install -U pip


## 8) Why `PIP FREEZE` failed but `pip freeze` worked

Linux commands are **case-sensitive**.

So:
- `PIP` ❌ is not the same as `pip` ✅

That’s why you got:
- `PIP: command not found`

✅ Correct command:


In [ ]:
pip freeze

## 9) Why `pip freeze` returned nothing (and why that’s OK)

In your log, after activation you ran:
- `pip freeze`
…and it printed **nothing**.

That can be **completely normal** because:
- You created a fresh venv
- You upgraded `pip`
- But you did not install any *project packages* yet

`pip freeze` lists installed packages. If none are installed, output can be empty.

### Try installing a package to see `pip freeze` output change:


In [ ]:
# Run in terminal:
pip install requests

# Now you should see requests (and dependencies) in freeze:
pip freeze


## 10) Best practice for projects: save dependencies to requirements.txt

If you want students to reproduce your environment:

✅ Export:


In [ ]:
# Run in terminal:
pip freeze > requirements.txt


### Recreate environment later (or on another machine)

✅ Typical workflow:


In [ ]:
# Run in terminal:
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip
pip install -r requirements.txt


## 11) Troubleshooting checklist (fast)

If things go wrong, these commands pinpoint the issue:

```bash
python3 --version
which python3

# If venv activated:
which python
which pip
python --version
pip --version

# If packages seem missing:
python -m pip freeze
```

### One golden rule
Prefer this form to avoid “wrong pip” issues:
```bash
python -m pip install <package>
python -m pip freeze
```


## 12) Your exact successful log (kept here for students)

Use this section as a “real-life example” to show students what success looks like and what errors mean.


```text
sudo apt install -y python3-venv
E: Package 'python3-venv' has no installation candidate

python3 --version
Python 3.12.3

sudo apt install -y python3.12-venv
E: Package 'python3.12-venv' has no installation candidate

cat /etc/os-release
Ubuntu 24.04.3 LTS (Noble Numbat)

sudo apt update
sudo apt install -y software-properties-common
sudo add-apt-repository -y universe
sudo apt update

sudo apt install -y python3-venv
# installs python3.12-venv + python3-venv successfully

python3 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip

PIP FREEZE
PIP: command not found

pip freeze
# (empty output is OK in a new venv)

```